# setup

In [83]:
import duckdb as dk
import yfinance as yf

# Composites
comp_hold  = r'holdings/*holdings.parquet'
comp_price_1h = r'composites/*1h_price.parquet'
comp_price_1d = r'composites/*1d_price.parquet'
comp_price_1wk = r'composites/*1wk_price.parquet'
comp_meta  = r'composites/*metadata.parquet'
# Constituents
consti_price_1h  = r'constituents/*1h_price.parquet'
consti_price_1d  = r'constituents/*1d_price.parquet'
consti_price_1wk = r'constituents/*1wk_price.parquet'
consti_meta  = r'constituents/*metadata.parquet'
consti_fin_ttm_income   = r'constituents/*ttm_income_financial.parquet'
consti_fin_ttm_cashflow = r'constituents/*ttm_cashflow_financial.parquet'
consti_fin_qtr_income   = r'constituents/*qtr_income_financial.parquet'
consti_fin_qtr_cashflow = r'constituents/*qtr_cashflow_financial.parquet'
consti_fin_qtr_assets   = r'constituents/*qtr_assets_financial.parquet'
consti_fin_dates = r'constituents/*dates_financial.parquet'

In [121]:
print(dk.sql("""
select columns('fiftyTwo')
from read_parquet('constituents/nvda_metadata.parquet', union_by_name=True)
""").fetchdf().T)

                                             0
fiftyTwoWeekLow                         151.49
fiftyTwoWeekHigh                        236.54
fiftyTwoWeekLowChange                43.479996
fiftyTwoWeekLowChangePercent          0.287016
fiftyTwoWeekRange              151.49 - 236.54
fiftyTwoWeekHighChange              -41.569992
fiftyTwoWeekHighChangePercent        -0.175742
fiftyTwoWeekChangePercent            21.862137


# comp summary

In [20]:
def get_etf_summary():
    query = dk.sql(f"""
        with 
        clean_list as (
            select
                etf
                , Name as name
                , Ticker as symbol
                , SEDOL as sedol
                , Weight as weight
            from read_parquet('{comp_hold}', union_by_name=True)
            where
                sedol != '-'
        )
        select
            etf
            , count(*) as count
            , round(sum(weight), 2) as weight
        from clean_list
        group by etf
        order by count desc
    """)
    return query

get_etf_summary()

┌─────────┬───────┬────────┐
│   etf   │ count │ weight │
│ varchar │ int64 │ double │
├─────────┼───────┼────────┤
│ SPY     │   503 │  99.99 │
│ KRE     │   161 │   99.8 │
│ XBI     │   152 │  99.86 │
│ XSW     │   132 │  99.96 │
│ KBE     │   103 │  99.79 │
│ XLI     │    80 │  99.92 │
│ XLF     │    76 │  99.84 │
│ XRT     │    75 │  99.83 │
│ XLK     │    74 │  99.93 │
│ XHE     │    67 │   99.9 │
│  ·      │     · │     ·  │
│  ·      │     · │     ·  │
│  ·      │     · │     ·  │
│ XTL     │    40 │  99.93 │
│ XME     │    39 │  99.99 │
│ XLP     │    35 │  99.64 │
│ XHB     │    35 │  99.94 │
│ XES     │    34 │  99.87 │
│ XLU     │    31 │  99.68 │
│ XLRE    │    31 │   99.5 │
│ XLB     │    26 │  99.86 │
│ XLC     │    23 │  99.87 │
│ XLE     │    21 │  99.83 │
└─────────┴───────┴────────┘
  30 rows        3 columns
  (20 shown)               

# comp holdings

In [173]:
def get_etf_holdings():
    query = dk.sql(f"""
        select
            etf
            , Ticker as symbol
            , Name as name
            --, SEDOL as sedol
            , round(Weight, 2) as weight
        from read_parquet('{comp_hold}', union_by_name=True)
        where
            sedol != '-'
    """)
    return query

get_etf_holdings()

┌─────────┬─────────┬──────────────────────────────┬────────┐
│   etf   │ symbol  │             name             │ weight │
│ varchar │ varchar │           varchar            │ double │
├─────────┼─────────┼──────────────────────────────┼────────┤
│ KBE     │ RKT     │ ROCKET COS INC CLASS A       │   1.11 │
│ KBE     │ TBBK    │ BANCORP INC/THE              │   1.05 │
│ KBE     │ NIC     │ NICOLET BANKSHARES INC       │   1.05 │
│ KBE     │ SFBS    │ SERVISFIRST BANCSHARES INC   │   1.03 │
│ KBE     │ UMBF    │ UMB FINANCIAL CORP           │   1.02 │
│ KBE     │ NMIH    │ NMI HOLDINGS INC             │   1.02 │
│ KBE     │ ASB     │ ASSOCIATED BANC CORP         │   1.02 │
│ KBE     │ AX      │ AXOS FINANCIAL INC           │   1.01 │
│ KBE     │ ESNT    │ ESSENT GROUP LTD             │   1.01 │
│ KBE     │ GBCI    │ GLACIER BANCORP INC          │   1.01 │
│  ·      │  ·      │          ·                   │     ·  │
│  ·      │  ·      │          ·                   │     ·  │
│  ·    

# comp aggs

In [170]:
dk.sql(f"""
with
consti_meta as (
    select
        symbol, sector, industry
        , marketCap as mcap
        , totalRevenue as revenue
        , sharesOutstanding * trailingEps as income
        , bookValue as equity
        , totalDebt as debt
        , freeCashflow as cashflow
        , currentPrice as close
        , fiftyTwoWeekHigh as yearhigh
        , fiftyTwoWeekLow as yearlow
        , allTimeHigh as alltimehigh
        , allTimeLow as alltimelow
        , fiftyDayAverage as ma50
        , twoHundredDayAverage as ma200
    from read_parquet('{consti_meta}', union_by_name=True)
),
etf_holdings as (
    with
    meta as (
        with
        prep_meta as (
            select *
                , ((close / yearhigh) - 1) * 100 as relative_yearhigh
                , ((close / alltimehigh) - 1) * 100 as relative_alltimehigh
            from consti_meta
        )
        select *
            , case when close > ma50 then 'above50' when close > ma200 then 'above200' end as trend
            , case when relative_yearhigh > -20 then 'bullish' else null end as yearpivot
            , case when relative_alltimehigh > -20 then 'bullish' else null end as alltimepivot
        from prep_meta
    )
    select
        etf
        , Ticker as symbol
        , Name as name
        --, SEDOL as sedol
        , round(Weight, 2) as weight
        , b.sector, industry
        , mcap, revenue, income, cashflow, equity, debt
        , trend, yearpivot, alltimepivot
    from read_parquet('{comp_hold}', union_by_name=True) a
    left join meta b on a.Ticker = b.symbol
    where
        sedol != '-'
)
select
    etf
    , count(*) as count
    , try_cast(sum(mcap) as bigint) as mcap
    , try_cast(sum(revenue) as bigint) as revenue
    , try_cast(sum(income) as bigint) as income
    , round((sum(income) / sum(revenue)) * 100, 2) as margin
    , round((sum(mcap) / sum(revenue)), 2) as ps
    , round((sum(mcap) / sum(income)), 2) as pe
    , round((count(*) filter (trend = 'above50')  / count(*)) * 100, 2) as breadth_ma50
    , round((count(*) filter (trend = 'above200') / count(*)) * 100, 2) as breadth_ma200
    , round((count(*) filter (yearpivot = 'bullish') / count(*)) * 100, 2) as breadth_yearpivot
    , round((count(*) filter (alltimepivot = 'bullish') / count(*)) * 100, 2) as breadth_alltimepivot    
from etf_holdings
group by etf
order by mcap desc
""")

┌─────────┬───────┬────────────────┬────────────────┬───────────────┬────────┬────────┬────────┬──────────────┬───────────────┬───────────────────┬──────────────────────┐
│   etf   │ count │      mcap      │    revenue     │    income     │ margin │   ps   │   pe   │ breadth_ma50 │ breadth_ma200 │ breadth_yearpivot │ breadth_alltimepivot │
│ varchar │ int64 │     int64      │     int64      │     int64     │ double │ double │ double │    double    │    double     │      double       │        double        │
├─────────┼───────┼────────────────┼────────────────┼───────────────┼────────┼────────┼────────┼──────────────┼───────────────┼───────────────────┼──────────────────────┤
│ SPY     │   503 │ 71185969545216 │ 18697525430016 │ 2250240294639 │  12.03 │   3.81 │  31.63 │        64.02 │          9.34 │             64.02 │                44.93 │
│ XLK     │    74 │ 24408749032448 │  2488148545536 │  667458332899 │  26.83 │   9.81 │  36.57 │         50.0 │         20.27 │             51.35

# consti indicators

In [82]:
st = 20
lt = st*3

def get_indicators():
    query = dk.sql(f"""
    with 
    master_file as (
        with
        previous_day as (
            select symbol, date
                , open, high, low, close
                , lag(open)  over (partition by symbol order by date asc) as pd_open
                , lag(high)  over (partition by symbol order by date asc) as pd_high
                , lag(low)   over (partition by symbol order by date asc) as pd_low
                , lag(close) over (partition by symbol order by date asc) as pd_close
                , lag(close, {st}) over (partition by symbol order by date) as pd_close_{st}
                , lag(close, {lt}) over (partition by symbol order by date) as pd_close_{lt}
                , volume, cast(volume * close as bigint) as value
            from read_parquet('{consti_price_1d}', union_by_name=True)
        ),
        change as (
            with 
            pre as (
                select *
                    , ((close / nullif(pd_close, 0)) - 1) * 100 as chg
                    , ((open  / nullif(pd_close, 0)) - 1) * 100 as gap
                    , ((close / nullif(pd_close_{st}, 0)) - 1) * 100 as chg_{st}
                    , ((close / nullif(pd_close_{lt}, 0)) - 1) * 100 as chg_{lt}
                    , case when close>pd_close then close-pd_close else 0 end as gain
                    , case when close<pd_close then abs(pd_close-close) else 0 end as loss
                from previous_day
            )
            select *
                , avg(gain) over st as avg_gain_{st}
                , avg(gain) over lt as avg_gain_{lt}
                , avg(loss) over st as avg_loss_{st}
                , avg(loss) over lt as avg_loss_{lt}
                , avg(volume) over st as avg_vol_{st}
                , avg(volume) over lt as avg_vol_{lt}
                , avg(value)  over st as avg_value_{st}
                , avg(value)  over lt as avg_value_{lt}
            from pre
            window
                st as (
                    partition by symbol
                    order by date rows
                    between {st-1} preceding and current row
                ),
                lt as (
                    partition by symbol
                    order by date rows
                    between {lt-1} preceding and current row
                )
        )
        select *
            , avg(chg)             over w AS mean_chg
            , stddev_pop(chg)      over w AS stddev_chg
            , avg(chg_{st})        over w AS mean_chg_{st}
            , stddev_pop(chg_{st}) over w AS stddev_chg_{st}
            , avg(chg_{lt})        over w AS mean_chg_{lt}
            , stddev_pop(chg_{lt}) over w AS stddev_chg_{lt}
        from change
        window w as (
            partition by symbol order by date
            rows between unbounded preceding and current row
        )
    ),
    indicators as (
        select *
            , volume / avg_vol_{st} as rvol
            , case when avg_loss_{st}=0 then 100 else 100-(100/(1+(avg_gain_{st}/avg_loss_{st}))) end as rsi_{st}
            , case when avg_loss_{lt}=0 then 100 else 100-(100/(1+(avg_gain_{lt}/avg_loss_{lt}))) end as rsi_{lt}
            , (chg - mean_chg) / stddev_chg as z_score
            , (chg_{st} - mean_chg_{st}) / stddev_chg_{st} as z_score_{st}
            , (chg_{lt} - mean_chg_{lt}) / stddev_chg_{lt} as z_score_{lt}
        from master_file
    )
    select symbol
        , try_cast(date as date) as date
        , try_cast(open as numeric(18,2)) as open
        , try_cast(high as numeric(18,2)) as high
        , try_cast(low as numeric(18,2)) as low
        , try_cast(close as numeric(18,2)) as close
        , try_cast(chg as numeric(18,2)) as chg
        , try_cast(rvol as numeric(18,2)) as rvol
        , try_cast(volume as bigint) as volume
        , try_cast(avg_vol_{st} as bigint) as avg_vol_{st}
        , try_cast(value as bigint) as value
        , try_cast(avg_value_{st} as bigint) as avg_value_{st}
        , try_cast(rsi_{st} as numeric(18,2)) as rsi_{st} 
        , try_cast(rsi_{lt} as numeric(18,2)) as rsi_{lt} 
        , try_cast(((rsi_{st} / rsi_{lt}) - 1) * 100 as numeric(18,2)) as rsi_trend
        , try_cast(z_score as numeric(18,2)) as z_score 
        , try_cast(z_score_{st} as numeric(18,2)) as z_score_{st} 
        , try_cast(z_score_{lt} as numeric(18,2)) as z_score_{lt}
    from indicators
    order by date desc, value desc
    """)
    return query

get_indicators()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────────┬───────────────┬───────────────┬───────────────┬───────────────┬───────────────┬───────────────┬───────────┬────────────┬─────────────┬──────────────┬───────────────┬───────────────┬───────────────┬───────────────┬───────────────┬───────────────┐
│ symbol  │    date    │     open      │     high      │      low      │     close     │      chg      │     rvol      │  volume   │ avg_vol_20 │    value    │ avg_value_20 │    rsi_20     │    rsi_60     │   rsi_trend   │    z_score    │  z_score_20   │  z_score_60   │
│ varchar │    date    │ decimal(18,2) │ decimal(18,2) │ decimal(18,2) │ decimal(18,2) │ decimal(18,2) │ decimal(18,2) │   int64   │   int64    │    int64    │    int64     │ decimal(18,2) │ decimal(18,2) │ decimal(18,2) │ decimal(18,2) │ decimal(18,2) │ decimal(18,2) │
├─────────┼────────────┼───────────────┼───────────────┼───────────────┼───────────────┼───────────────┼───────────────┼───────────┼────────────┼─────────────┼──────────────┼─────────────

# consti financials

In [40]:
print(dk.sql("""
select *
from read_parquet('constituents/nvda_ttm_cashflow_financial.parquet', union_by_name=True)
""").fetchdf().columns.tolist())

def get_ttm_income():

    financials = dk.sql("""
    select
        symbol
        ,__index_level_0__ as date
        , try_cast("Total Revenue" as bigint) as revenue
        , try_cast("Gross Profit" as bigint) as gross_profit
        , try_cast("Operating Income" as bigint) as operating_income
        , try_cast("Net Income" as bigint) as net_income
        , try_cast("EBITDA" as bigint) as ebitda
        , try_cast("Research And Development" as bigint) as research
    from read_parquet('constituents/*ttm_income_financial.parquet', union_by_name=True)
    order by revenue desc
    """)
    return financials

def get_qtr_cashflow():

    financials = dk.sql("""
    with
    clean as (
        with pre as (
            select
                symbol
                , try_cast(__index_level_0__ as date) as date
                , try_cast("Free Cash Flow" as bigint) as free_cashflow
                , try_cast("Capital Expenditure" as bigint) as capex
                , try_cast("Financing Cash Flow" as bigint) as financing_casflow
                , try_cast("Investing Cash Flow" as bigint) as investing_casflow
                , try_cast("Operating Cash Flow" as bigint) as operating_casflow
                , round(try_cast("Operating Cash Flow" as bigint) / try_cast(abs("Capital Expenditure") as bigint), 2) as cashflow_ratio
            from read_parquet('constituents/*_qtr_cashflow_financial.parquet', union_by_name=True)
        )
        select
            *
            , round(((cashflow_ratio / lag(cashflow_ratio) over (partition by symbol order by date asc)) - 1) * 100, 2) as ratio_chg
            , case
                when cashflow_ratio > 2 then 'generation'
                when cashflow_ratio < 1 then 'investing' else ''
                    end as phase
        from pre
    )
    select *
    from clean
    where date = (select max(date) from clean)
    order by free_cashflow desc
    """)
    return financials

get_qtr_cashflow()

# ticker = 'NVDA'
# ttm_income = yf.Ticker(ticker).income_stmt.T
# ttm_income.reset_index()
# ttm_income.insert(0, 'symbol', ticker)
# ttm_income

['symbol', 'Free Cash Flow', 'Repurchase Of Capital Stock', 'Capital Expenditure', 'End Cash Position', 'Beginning Cash Position', 'Changes In Cash', 'Financing Cash Flow', 'Cash Flow From Continuing Financing Activities', 'Net Other Financing Charges', 'Proceeds From Stock Option Exercised', 'Cash Dividends Paid', 'Common Stock Dividend Paid', 'Net Common Stock Issuance', 'Common Stock Payments', 'Investing Cash Flow', 'Cash Flow From Continuing Investing Activities', 'Net Investment Purchase And Sale', 'Sale Of Investment', 'Purchase Of Investment', 'Net Business Purchase And Sale', 'Purchase Of Business', 'Net PPE Purchase And Sale', 'Purchase Of PPE', 'Operating Cash Flow', 'Cash Flow From Continuing Operating Activities', 'Change In Working Capital', 'Change In Other Current Liabilities', 'Change In Payables And Accrued Expense', 'Change In Accrued Expense', 'Change In Payable', 'Change In Account Payable', 'Change In Prepaid Assets', 'Change In Inventory', 'Change In Receivables'

┌─────────┬────────────┬───────────────┬──────────────┬───────────────────┬───────────────────┬───────────────────┬────────────────┬───────────┬────────────┐
│ symbol  │    date    │ free_cashflow │    capex     │ financing_casflow │ investing_casflow │ operating_casflow │ cashflow_ratio │ ratio_chg │   phase    │
│ varchar │    date    │     int64     │    int64     │       int64       │       int64       │       int64       │     double     │  double   │  varchar   │
├─────────┼────────────┼───────────────┼──────────────┼───────────────────┼───────────────────┼───────────────────┼────────────────┼───────────┼────────────┤
│ MU      │ 2026-05-31 │   17562000000 │  -7826000000 │       -4734000000 │       -9569000000 │       25388000000 │           3.24 │     74.19 │ generation │
│ ACN     │ 2026-05-31 │    3599988000 │   -186224000 │       -1766145000 │       -1206553000 │        3786212000 │          20.33 │    -20.27 │ generation │
│ ADBE    │ 2026-05-31 │    2107000000 │    -5800000

# consti options

In [85]:
import duckdb as dk
calls = r"C:\GitHub\rlp-jym-projects\spdr-pipeline\pipeline\uploads\consti_options_calls.parquet"
puts = r"C:\GitHub\rlp-jym-projects\spdr-pipeline\pipeline\uploads\consti_options_puts.parquet"


dk.sql(f"""
with 
joined as (
    select * from read_parquet('{calls}')
    union all
    select * from read_parquet('{puts}')
),
pre as (
    select 
        * exclude (inTheMoney, contractSize, currency, contractSymbol, change)
        , impliedVolatility * (volume / sum(volume) over (partition by symbol, type)) as volWeightIV
        , impliedVolatility * (openInterest / sum(openInterest) over (partition by symbol, type)) as oiWeightIV
    from joined
),
agg as (
select
    symbol, type
    , sum(volume) as vol
    , sum(openInterest) as oi
    , round(sum(volWeightIV), 2) as volWeightIV
    , round(sum(oiWeightIV), 2) as oiWeightIV
from pre
group by symbol, type
order by symbol, type
),
flat as (
select
    symbol
    , sum(vol) as vol
    , sum(oi) as oi
    , max(case when type = 'calls' then oiWeightIV end) as callsIV
    , max(case when type = 'puts' then oiWeightIV end) as putsIV
from agg
group by symbol
),
ratio as (
select
    symbol
    , cast(oi as int) as totalOI
    , callsIV, putsIV
    , round(callsIV/putsIV, 2) as ratio
from flat
where true
    and ratio != 'nan'
    and ratio != 'inf'
order by totalOI desc
)
select *
    , case
        when ratio < 0.5 then 'high short'
        when ratio > 1 then 'unusual'
            else '' end as signal
from ratio
where true
    and totalOI > 100e3
""")
# ratio threshold and minimum oi are arbitrary
# to be validated with more data inputs
# still day 1 of options data gathering

┌─────────┬─────────┬─────────┬────────┬────────┬────────────┐
│ symbol  │ totalOI │ callsIV │ putsIV │ ratio  │   signal   │
│ varchar │  int32  │ double  │ double │ double │  varchar   │
├─────────┼─────────┼─────────┼────────┼────────┼────────────┤
│ MU      │  452540 │    0.92 │    3.3 │   0.28 │ high short │
│ MSTR    │  433872 │    1.27 │   1.48 │   0.86 │            │
│ PLTR    │  233928 │     0.7 │   0.78 │    0.9 │            │
│ INTC    │  233444 │    0.95 │    1.4 │   0.68 │            │
│ MARA    │  208998 │    0.97 │    1.0 │   0.97 │            │
│ NVDA    │  201459 │    0.44 │   0.55 │    0.8 │            │
│ SMCI    │  167914 │    1.16 │   1.24 │   0.94 │            │
│ NFLX    │  153820 │    0.56 │   0.42 │   1.33 │ unusual    │
│ MRVL    │  147015 │    1.06 │   1.65 │   0.64 │            │
│ AMD     │  139653 │    0.82 │   1.29 │   0.64 │            │
│ HOOD    │  125252 │     0.8 │   1.07 │   0.75 │            │
│ WULF    │  121412 │    1.87 │    1.2 │   1.56 │ unusu

# yfinance resources

In [44]:
# HOLDINGS
    # insider_purchases
    # insider_transactions
    # insider_roster_holders
    # institutional_holders
    # mutualfund_holders

# ANALYSIS
    # revenue_estimate
    # earnings_estimate
    # growth_estimates
    # eps_revisions
    # eps_trend
    # upgrades_downgrades
    # recommendations
    # analyst_price_targets

# STOCKS
    # dividends
    # splits

# SECTORS
    # keys
        # basic-materials
        # communication-services
        # consumer-cyclical
        # consumer-defensive
        # energy
        # financial-services
        # healthcare
        # industrials
        # real-estate
        # technology
        # utilities
    # attributes
        # industries
        # name
        # overview
        # research_reports
        # symbol
        # ticker
        # top_companies
        # top_etfs
        # top_mutual_funds
        # industry-specific
           # sector_name
           # top_growth_companies
           # top_performing_companies

ticker = 'NVDA'
yf.Ticker(ticker).ttm_income_stmt.T.reset_index().to_parquet(f'{ticker}_ttm_income.parquet')
yf.Ticker(ticker).ttm_cashflow.T.reset_index().to_parquet(f'{ticker}_ttm_cashflow.parquet')
yf.Ticker(ticker).quarterly_income_stmt.T.reset_index().to_parquet(f'{ticker}_qtr_income.parquet')
yf.Ticker(ticker).quarterly_cashflow.T.reset_index().to_parquet(f'{ticker}_qtr_cashflow.parquet')
yf.Ticker(ticker).quarterly_balance_sheet.T.reset_index().to_parquet(f'{ticker}_qtr_assets.parquet')
yf.Ticker(ticker).earnings_dates.reset_index().to_parquet(f'{ticker}_dates.parquet')

In [ ]:
# ticker_list = [
#     # state street etfs
#     'XLC', 'XLY', 'XLP', 'XLE', 'XLF', 'XLV', 'XLI', 'XLB', 'XLRE', 'XAR',
#     'KBE', 'XBI', 'KCE', 'XHE', 'XHS', 'XHB', 'KIE', 'XME', 'XES', 'XOP', 
#     'XPH', 'KRE', 'XRT', 'XSD', 'XSW', 'XTL', 'XTN', 'XLK', 'XLU', 'SPY'
# ]
# # ticker_list = ['NVDA', 'TSLA', 'INTC']

# options = pd.DataFrame()
# for ticker in ticker_list:
#     optionchaincalls = yf.Ticker(ticker).option_chain().calls
#     optionchainputs = yf.Ticker(ticker).option_chain().puts
    
#     optionsx = duckdb.sql(f"""
#         with
#         agg as (
#             with 
#             merged as (
#                 select 'calls' as type, * from optionchaincalls
#                 union all
#                 select 'puts' as type, * from optionchainputs
#             ),
#             calls as (
#                 with totals as (
#                     select
#                         type,
#                         impliedVolatility as IV, 
#                         volume, openInterest as OI, 
#                         sum(volume)       over (partition by type) as totalVolume,
#                         sum(openInterest) over (partition by type) as totalInterest
#                     from merged
#                 )
#                 select *,
#                     volume / totalVolume as weightedVol,
#                     OI     / totalInterest as weightedOI 
#                 from totals
#             ),
#             weighted as (
#                 select *,
#                     IV * weightedVol as volWeightIV,
#                     IV * weightedOI as OIWeightIV
#                 from calls
#             )
#             select
#                 type,
#                 sum(volume) as volume,
#                 sum(OI) as OI,
#                 round(sum(volWeightIV), 2) as volWeightIV,
#                 round(sum(OIWeightIV), 2) as OIWeightIV
#             from weighted
#             group by type
#         ),
#         ratio as (
#             select
#                 '{ticker}' as ticker,
#                 round(
#                     max(case when type = 'calls' then volume end) / 
#                     max(case when type = 'puts'  then volume end), 2) as volume,
#                 round(
#                     max(case when type = 'calls' then OI end) / 
#                     max(case when type = 'puts'  then OI end), 2) as OI,
#                 round(
#                     max(case when type = 'calls' then volWeightIV end) / 
#                     max(case when type = 'puts'  then volWeightIV end), 2) as volWeightIV,
#                 round(
#                     max(case when type = 'calls' then OIWeightIV end) / 
#                     max(case when type = 'puts'  then OIWeightIV end), 2) as OIWeightIV
#             from agg
#         )
#         select * from ratio
#     """).fetchdf()

#     options = pd.concat([options, optionsx], ignore_index=True)   

# options

In [14]:
import duckdb
duckdb.sql("""
select *
from read_parquet('C:\GitHub\rlp-jym-projects\spdr-pipeline\pipeline\holdings\*.parquet', union_by_name=True) 
""")

<>:4: SyntaxWarning: "\G" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\G"? A raw string is also an option.
<>:4: SyntaxWarning: "\G" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\G"? A raw string is also an option.
C:\Users\ralph\AppData\Local\Temp\ipykernel_23864\2819547204.py:4: SyntaxWarning: "\G" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\G"? A raw string is also an option.
  from read_parquet('C:\GitHub\rlp-jym-projects\spdr-pipeline\pipeline\holdings\*.parquet', union_by_name=True)


IOException: IO Error: No files found that match the pattern "C:\GitHublp-jym-projects\spdr-pipeline\pipeline\holdings\*.parquet"

# end